# 02 — MetroPT-3 Exploratory Data Analysis

Descriptive EDA on the P1-002 processed MetroPT-3 outputs plus a chunked
re-scan of raw signals (the 235 MB readings file is never fully loaded).
No SPC, no RCA, no modeling. Key numbers go to
`reports/metropt3_eda_summary.json`, figures to `reports/figures/metropt3_*.png`.

Documented-data note: the CSV itself is unlabeled. Failure intervals and the
24 h pre-failure windows are transcribed UCI documentation, and the operating
state is a project-assumption rule on motor current — neither is observed ground truth.

In [1]:
from pathlib import Path

import json
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

candidate = Path.cwd()
while not (candidate / "pyproject.toml").exists():
    candidate = candidate.parent
ROOT = candidate
sys.path.insert(0, str(ROOT))

from src.config_loader import load_config

config = load_config()
mp = config["metropt3"]
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

ingestion = json.loads((ROOT / mp["outputs"]["ingestion_summary"]).read_text(encoding="utf-8"))
intervals = pd.read_csv(ROOT / mp["outputs"]["failure_intervals"])
print("ingestion total_rows:", ingestion["total_rows"])
print(intervals.to_dict(orient="records"))

ingestion total_rows: 1516948
[{'event_id': 1, 'kind': 'Air leak', 'window_start': '2020-04-18 00:00:00', 'window_end': '2020-04-18 23:59:59', 'readings_in_window': 8663, 'prefailure_readings': 6301}, {'event_id': 2, 'kind': 'Air leak', 'window_start': '2020-05-29 23:30:00', 'window_end': '2020-05-30 06:00:00', 'readings_in_window': 2360, 'prefailure_readings': 7466}, {'event_id': 3, 'kind': 'Air leak', 'window_start': '2020-06-05 10:00:00', 'window_end': '2020-06-07 14:30:00', 'readings_in_window': 17315, 'prefailure_readings': 8716}, {'event_id': 4, 'kind': 'Air leak', 'window_start': '2020-07-15 14:30:00', 'window_end': '2020-07-15 19:00:00', 'readings_in_window': 1622, 'prefailure_readings': 6185}]


## 1. Shape, timestamp range, reconciliation with P1-002

In [2]:
ANALOG = ["TP2", "TP3", "H1", "DV_pressure", "Reservoirs", "Oil_temperature", "Motor_current"]
DIGITAL = ["COMP", "DV_eletric", "Towers", "MPG", "LPS", "Pressure_switch", "Oil_level", "Caudal_impulses"]
EVENTS = [(e["id"], pd.Timestamp(e["start"]), pd.Timestamp(e["end"]), e["kind"])
          for e in mp["failure_events"]]
PREF_H = float(mp["prefailure_hours"])

assert ingestion["total_rows"] == ingestion["expected_rows"] == 1516948
assert intervals["readings_in_window"].sum() == ingestion["failure_window_rows"] == 29960
assert intervals["prefailure_readings"].sum() == ingestion["prefailure_window_rows"] == 28668
print("P1-002 totals reconcile: rows=1516948 failure=29960 prefailure=28668")

P1-002 totals reconcile: rows=1516948 failure=29960 prefailure=28668


## 2. Chunked signal scan: ranges, nulls, cadence, gaps, event neighborhoods

In [3]:
source = ROOT / config["paths"]["raw"] / mp["raw_filenames"]["readings"]
assert source.is_file(), source

GAP_S = 120
FOCUS = ["TP2", "TP3", "Motor_current", "Oil_temperature"]
NBHD_H = 6.0

sums = {s: 0.0 for s in ANALOG}
sumsq = {s: 0.0 for s in ANALOG}
mins = {s: float("inf") for s in ANALOG}
maxs = {s: float("-inf") for s in ANALOG}
null_total = 0
digital_ok = True
n = 0
prev_tail = None
diffs = []
gaps = []
win = {(eid, zone): {"n": 0, **{s: 0.0 for s in FOCUS}}
       for eid, _, _, _ in EVENTS for zone in ("pre6h", "during", "post6h")}
sample_rows = []
SAMPLE_EVERY = 30

for chunk in pd.read_csv(source, chunksize=100_000):
    ts = pd.to_datetime(chunk["timestamp"], errors="raise")
    if ts.isna().any():
        raise ValueError("unparsable timestamp in scan")
    if prev_tail is not None and ts.iloc[0] < prev_tail:
        raise ValueError("timestamps not non-decreasing")
    step = (ts.diff().dropna().dt.total_seconds())
    diffs.extend(step.tolist())
    over = step[step > GAP_S]
    for idx, secs in over.items():
        gaps.append({"end": str(ts.loc[idx]), "gap_seconds": float(secs)})
    prev_tail = ts.iloc[-1]

    mat = chunk[ANALOG].apply(pd.to_numeric, errors="raise")
    null_total += int(chunk.isna().sum().sum())
    for s in ANALOG:
        v = mat[s].to_numpy(dtype="float64")
        sums[s] += float(v.sum())
        sumsq[s] += float((v ** 2).sum())
        mins[s] = min(mins[s], float(v.min()))
        maxs[s] = max(maxs[s], float(v.max()))
    dmat = chunk[DIGITAL].apply(pd.to_numeric, errors="raise")
    if not ((dmat == 0) | (dmat == 1)).all().all():
        digital_ok = False

    for eid, start, end, _ in EVENTS:
        pre0, post1 = start - pd.Timedelta(hours=NBHD_H), end + pd.Timedelta(hours=NBHD_H)
        for zone, lo, hi in (("pre6h", pre0, start), ("during", start, end), ("post6h", end, post1)):
            mask = (ts >= lo) & (ts < hi) if zone != "during" else (ts >= lo) & (ts <= hi)
            sel = mat.loc[mask.index[mask], FOCUS]
            win[(eid, zone)]["n"] += int(len(sel))
            for s in FOCUS:
                win[(eid, zone)][s] += float(sel[s].sum())

    take = ((np.arange(len(chunk)) + n) % SAMPLE_EVERY == 0)
    sample_rows.append(chunk.loc[take, ANALOG + DIGITAL])
    n += len(chunk)

sample = pd.concat(sample_rows, ignore_index=True)
print(f"scanned rows={n} null_cells={null_total} digital_binary={digital_ok} sample_rows={len(sample)}")
assert n == 1516948 and null_total == 0 and digital_ok

scanned rows=1516948 null_cells=0 digital_binary=True sample_rows=50565


## 3. Sampling cadence and logging gaps

In [4]:
diffs = np.array(diffs)
print("cadence seconds: median=%.1f p99=%.1f max=%.1f" % (
    float(np.median(diffs)), float(np.quantile(diffs, 0.99)), float(diffs.max())))
print(f"gaps > {GAP_S}s: {len(gaps)}")
gaps_sorted = sorted(gaps, key=lambda g: g["gap_seconds"], reverse=True)[:15]
for g in gaps_sorted:
    print(g)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.clip(diffs, 0, 300), bins=60)
ax.set_xlabel("seconds between consecutive readings (clipped at 300)")
ax.set_ylabel("count")
ax.set_title("MetroPT-3 sampling cadence")
fig.tight_layout()
fig.savefig(FIG_DIR / "metropt3_cadence_hist.png", dpi=120)
plt.close(fig)

cadence seconds: median=10.0 p99=12.0 max=172918.0
gaps > 120s: 327
{'end': '2020-04-27 01:12:49', 'gap_seconds': 172918.0}
{'end': '2020-06-28 23:07:43', 'gap_seconds': 130476.0}
{'end': '2020-03-01 04:00:09', 'gap_seconds': 100981.0}
{'end': '2020-08-05 08:23:01', 'gap_seconds': 88833.0}
{'end': '2020-05-25 01:14:14', 'gap_seconds': 88491.0}
{'end': '2020-07-08 15:20:51', 'gap_seconds': 86160.0}
{'end': '2020-08-23 18:51:01', 'gap_seconds': 85157.0}
{'end': '2020-05-10 22:48:58', 'gap_seconds': 80261.0}
{'end': '2020-06-08 11:48:04', 'gap_seconds': 77305.0}
{'end': '2020-04-02 09:59:17', 'gap_seconds': 74617.0}
{'end': '2020-03-28 23:05:32', 'gap_seconds': 51353.0}
{'end': '2020-07-14 21:28:05', 'gap_seconds': 51088.0}
{'end': '2020-05-01 12:33:38', 'gap_seconds': 46455.0}
{'end': '2020-08-03 09:01:16', 'gap_seconds': 44806.0}
{'end': '2020-05-17 23:10:01', 'gap_seconds': 43585.0}


## 4. Event 3 logging gap (observed, not interpolated)

In [5]:
e3 = next(e for e in EVENTS if e[0] == 3)
nominal = (e3[2] - e3[1]).total_seconds() / 10.0
actual = int(intervals.loc[intervals["event_id"] == 3, "readings_in_window"].iloc[0])
e3_gaps = [g for g in gaps if e3[1] <= pd.Timestamp(g["end"]) <= e3[2] + pd.Timedelta(seconds=GAP_S)]
print(f"event3 window_hours={(e3[2]-e3[1]).total_seconds()/3600:.1f} "
      f"nominal_rows~{nominal:.0f} actual_rows={actual} deficit={nominal-actual:.0f}")
print(f"gaps > {GAP_S}s intersecting event 3: {len(e3_gaps)}, "
      f"longest={max((g['gap_seconds'] for g in e3_gaps), default=0):.0f}s")
print("Treatment: observed logging/data-availability gap. No interpolation; "
      "downstream analysis must tolerate missing coverage inside failure windows.")

event3 window_hours=52.5 nominal_rows~18900 actual_rows=17315 deficit=1585
gaps > 120s intersecting event 3: 2, longest=13622s
Treatment: observed logging/data-availability gap. No interpolation; downstream analysis must tolerate missing coverage inside failure windows.


## 5. Sensor distributions (deterministic systematic sample)

In [6]:
stats = {}
for s in ANALOG:
    v = sample[s].to_numpy(dtype="float64")
    stats[s] = {"min": float(v.min()), "max": float(v.max()),
                "mean": float(v.mean()), "std": float(v.std()),
                "full_min": mins[s], "full_max": maxs[s],
                "full_mean": sums[s] / n}
    print(s, {k: round(x, 4) for k, x in stats[s].items()})
print("digital active rates:", {s: round(float(sample[s].mean()), 4) for s in DIGITAL})

fig, axes = plt.subplots(4, 2, figsize=(10, 10))
for ax, s in zip(axes.flat, ANALOG):
    ax.hist(sample[s], bins=60)
    ax.set_title(s)
    ax.tick_params(labelsize=7)
axes.flat[-1].axis("off")
fig.suptitle("MetroPT-3 analog sensor distributions (every-30th-row sample)")
fig.tight_layout()
fig.savefig(FIG_DIR / "metropt3_analog_hist.png", dpi=120)
plt.close(fig)

TP2 {'min': -0.03, 'max': 10.606, 'mean': 1.3555, 'std': 3.2385, 'full_min': -0.032, 'full_max': 10.676, 'full_mean': 1.3678}
TP3 {'min': 1.004, 'max': 10.302, 'mean': 8.9833, 'std': 0.6382, 'full_min': 0.73, 'full_max': 10.302, 'full_mean': 8.9846}
H1 {'min': -0.032, 'max': 10.244, 'mean': 7.582, 'std': 3.3213, 'full_min': -0.036, 'full_max': 10.288, 'full_mean': 7.5682}
DV_pressure {'min': -0.032, 'max': 7.98, 'mean': 0.0555, 'std': 0.3797, 'full_min': -0.032, 'full_max': 9.844, 'full_mean': 0.056}
Reservoirs {'min': 0.986, 'max': 10.3, 'mean': 8.9839, 'std': 0.6374, 'full_min': 0.712, 'full_max': 10.3, 'full_mean': 8.9852}
Oil_temperature {'min': 20.6, 'max': 89.0, 'mean': 62.6421, 'std': 6.5067, 'full_min': 15.4, 'full_max': 89.05, 'full_mean': 62.6442}
Motor_current {'min': 0.02, 'max': 7.27, 'mean': 2.0468, 'std': 2.2979, 'full_min': 0.02, 'full_max': 9.295, 'full_mean': 2.0502}
digital active rates: {'COMP': 0.8377, 'DV_eletric': 0.159, 'Towers': 0.9207, 'MPG': 0.8335, 'LPS': 0.

## 6. Operating-state, failure-window, pre-failure distributions

In [7]:
states = pd.read_csv(ROOT / mp["outputs"]["state_periods"])
print("state row counts (P1-002):", ingestion["state_counts"])
print(f"state periods: {len(states)}")
print("failure rows:", ingestion["failure_window_rows"], "| pre-failure rows:", ingestion["prefailure_window_rows"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
counts = pd.Series(ingestion["state_counts"])
axes[0].bar(counts.index, counts.values)
axes[0].set_title("rows per operating state")
axes[0].tick_params(axis="x", rotation=30)
axes[1].bar(intervals["event_id"].astype(str),
            intervals["readings_in_window"], label="in failure")
axes[1].bar(intervals["event_id"].astype(str),
            intervals["prefailure_readings"], bottom=intervals["readings_in_window"],
            label="pre-failure 24h")
axes[1].set_title("rows per failure event")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "metropt3_windows_bar.png", dpi=120)
plt.close(fig)

state row counts (P1-002): {'load': 201579, 'off': 829124, 'offloaded': 486201, 'starting': 44}
state periods: 70620
failure rows: 29960 | pre-failure rows: 28668


## 7. Sensor behavior around documented failures (6 h pre / during / 6 h post)

In [8]:
nbhd = {}
for eid, start, end, kind in EVENTS:
    nbhd[str(eid)] = {}
    for zone in ("pre6h", "during", "post6h"):
        w = win[(eid, zone)]
        nbhd[str(eid)][zone] = {"n": w["n"],
            **{s: (w[s] / w["n"] if w["n"] else None) for s in FOCUS}}
    row = {z: {s: (None if nbhd[str(eid)][z][s] is None else round(nbhd[str(eid)][z][s], 3))
               for s in FOCUS} for z in ("pre6h", "during", "post6h")}
    print(f"event {eid} ({kind}) n=", {z: nbhd[str(eid)][z]["n"] for z in row})
    print("  means:", row)

event 1 (Air leak) n= {'pre6h': 1790, 'during': 8663, 'post6h': 2018}
  means: {'pre6h': {'TP2': -0.018, 'TP3': 8.248, 'Motor_current': 0.04, 'Oil_temperature': 49.45}, 'during': {'TP2': 8.518, 'TP3': 8.812, 'Motor_current': 5.604, 'Oil_temperature': 74.131}, 'post6h': {'TP2': 3.724, 'TP3': 8.904, 'Motor_current': 3.327, 'Oil_temperature': 65.594}}
event 2 (Air leak) n= {'pre6h': 2179, 'during': 2360, 'post6h': 2080}
  means: {'pre6h': {'TP2': 1.261, 'TP3': 8.97, 'Motor_current': 1.92, 'Oil_temperature': 64.28}, 'during': {'TP2': 8.175, 'TP3': 8.402, 'Motor_current': 5.566, 'Oil_temperature': 75.871}, 'post6h': {'TP2': 0.961, 'TP3': 9.01, 'Motor_current': 1.855, 'Oil_temperature': 64.886}}
event 3 (Air leak) n= {'pre6h': 2179, 'during': 17315, 'post6h': 0}
  means: {'pre6h': {'TP2': 1.258, 'TP3': 9.026, 'Motor_current': 2.027, 'Oil_temperature': 64.218}, 'during': {'TP2': 7.892, 'TP3': 8.018, 'Motor_current': 5.494, 'Oil_temperature': 75.517}, 'post6h': {'TP2': None, 'TP3': None, 'Moto

## 8. Timeline: daily volume with operating states and failure spans

In [9]:
daily_states = (pd.read_csv(ROOT / mp["outputs"]["processed_readings"],
                            usecols=["timestamp", "operating_state"])
                  .assign(timestamp=lambda d: pd.to_datetime(d["timestamp"]),
                          date=lambda d: d["timestamp"].dt.date)
                  .groupby(["date", "operating_state"]).size().unstack(fill_value=0))
print("timeline days:", len(daily_states))

fig, ax = plt.subplots(figsize=(11, 4))
bottom = np.zeros(len(daily_states))
for state in ["off", "offloaded", "load", "starting"]:
    vals = daily_states[state].to_numpy() if state in daily_states else np.zeros(len(daily_states))
    ax.bar(daily_states.index.astype(str), vals, bottom=bottom, label=state)
    bottom += vals
for eid, start, end, _ in EVENTS:
    ax.axvspan(str(start.date()), str(end.date()), color="red", alpha=0.25)
ax.set_title("Daily readings by operating state (red = documented failure spans)")
ax.tick_params(axis="x", labelsize=5)
for label in ax.get_xticklabels()[::12]:
    label.set_visible(True)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "metropt3_timeline.png", dpi=120)
plt.close(fig)

timeline days: 212


## 9. Obvious anomalies (IQR rule on the systematic sample)

In [10]:
q1, q3 = sample[ANALOG].quantile(0.25), sample[ANALOG].quantile(0.75)
iqr = (q3 - q1).replace(0, float("nan"))
out_frac = ((sample[ANALOG] < (q1 - 1.5 * iqr)) | (sample[ANALOG] > (q3 + 1.5 * iqr))).mean()
print({s: round(float(v), 5) for s, v in out_frac.sort_values(ascending=False).items()})

{'TP2': 0.17591, 'H1': 0.15756, 'DV_pressure': 0.0739, 'Oil_temperature': 0.00352, 'TP3': 0.00295, 'Reservoirs': 0.00293, 'Motor_current': 0.0}


## 10. Evidence export and reconciliation

In [11]:
event_rows = {str(r["event_id"]): {"kind": r["kind"], "window_start": r["window_start"],
    "window_end": r["window_end"], "readings_in_window": int(r["readings_in_window"]),
    "prefailure_readings": int(r["prefailure_readings"]),
    "window_hours": round((pd.Timestamp(r["window_end"]) - pd.Timestamp(r["window_start"])).total_seconds() / 3600, 2)}
    for r in intervals.to_dict(orient="records")}

eda = {
    "total_rows": n,
    "timestamp_min": ingestion["timestamp_min"],
    "timestamp_max": ingestion["timestamp_max"],
    "cadence_seconds": {"median": float(np.median(diffs)), "p99": float(np.quantile(diffs, 0.99)),
                        "max": float(diffs.max())},
    "gap_count_gt_120s": len(gaps),
    "top_gaps": gaps_sorted,
    "null_cells": null_total,
    "digital_binary": digital_ok,
    "analog_stats": stats,
    "digital_active_rates": {s: float(sample[s].mean()) for s in DIGITAL},
    "state_counts": {k: int(v) for k, v in ingestion["state_counts"].items()},
    "state_period_count": int(len(states)),
    "failure_window_rows": ingestion["failure_window_rows"],
    "prefailure_window_rows": ingestion["prefailure_window_rows"],
    "events": event_rows,
    "event3_gap_note": ("Event 3 (Jun 5-7) actual rows %d vs ~%.0f nominal at 10 s cadence; "
                        "observed logging/data-availability gap, no interpolation." % (actual, nominal)),
    "event3_internal_gaps_gt_120s": len(e3_gaps),
    "neighborhood_means_6h": nbhd,
    "outlier_fraction_iqr_sample": {s: float(v) for s, v in out_frac.items()},
    "systematic_sample": "every 30th source row, deterministic, no RNG",
    "figures": ["metropt3_cadence_hist.png", "metropt3_analog_hist.png",
                "metropt3_windows_bar.png", "metropt3_timeline.png"],
}
out = ROOT / "reports" / "metropt3_eda_summary.json"
out.write_text(json.dumps(eda, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print("wrote", out)

wrote C:\Users\luqma\OneDrive\Desktop\Data Analyst Project\Production Yield Loss and OEE Early\reports\metropt3_eda_summary.json


## Findings

- 1,516,948 rows, zero nulls, binary digitals, ~10 s cadence with sparse long gaps.
- Compressor mostly off/offloaded; `starting` almost absent (44 rows).
- Four air-leak windows captured (29,960 rows) plus 28,668 pre-failure rows.
- Event 3 has an observed logging gap (~1.6k rows below nominal) — documented,
  not filled. Later phases must tolerate missing in-window coverage.
- No SPC, RCA, or modeling performed here by design.